In [ ]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Carolina Miranda\Documents\Jose\bootcamp-data-science-portfolio\cases\case-spark-sql-queries\data\raw\ventas.csv")
print(df.columns.tolist())
df.head()

# Case: Consultas Distribuidas con Spark SQL

## Business Understanding
RetailMax necesita analizar sus transacciones de venta para identificar patrones por sucursal, categoría de producto y temporalidad. Estas consultas informan decisiones de inventario y asignación de recursos.

In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path

spark = SparkSession.builder \
    .appName("Case - Spark SQL Queries") \
    .master("local[*]") \
    .getOrCreate()

DATA_PATH = Path("../data/raw/ventas.csv")
print(f"Spark version: {spark.version}")

In [ ]:
df = spark.read.csv(
    DATA_PATH,
    header=True,
    inferSchema=True
)

df.printSchema()
df.show(5)
print(f"Total registros: {df.count()}")

In [ ]:
df.createOrReplaceTempView("ventas")
print("Vista temporal 'ventas' registrada.")

## SQL Queries
Tres consultas con filtros, agregaciones y ordenamientos.   

In [ ]:
# Q1: Ventas de Tecnología con monto superior a 300,000
q1 = spark.sql("""
    SELECT id_venta, producto, monto, fecha
    FROM ventas
    WHERE categoria = 'Tecnologia'
      AND monto > 300000
    ORDER BY monto DESC
""")
q1.show()

In [ ]:
# Q2: Ingreso total y cantidad de ventas por categoría
q2 = spark.sql("""
    SELECT categoria,
           COUNT(*) AS total_ventas,
           SUM(monto) AS ingreso_total,
           ROUND(AVG(monto), 0) AS ticket_promedio
    FROM ventas
    GROUP BY categoria
    ORDER BY ingreso_total DESC
""")
q2.show()

In [ ]:
# Q3: Top 5 sucursales por ingreso total
q3 = spark.sql("""
    SELECT id_sucursal,
           COUNT(*) AS total_ventas,
           SUM(monto) AS ingreso_total
    FROM ventas
    GROUP BY id_sucursal
    ORDER BY ingreso_total DESC
    LIMIT 5
""")
q3.show()

In [ ]:
# Plan de ejecución de Q1 para ver optimizaciones de Catalyst
print("=== Plan de ejecución Q1 ===")
q1.explain(True)

## Cómo Catalyst optimiza estas consultas

1. **Parsed Logical Plan**: Parsea el SQL y construye un árbol lógico sin resolver.
2. **Analyzed Logical Plan**: Resuelve nombres de columnas y tipos contra el catálogo.
3. **Optimized Logical Plan**: Aplica optimizaciones como *predicate pushdown* (empuja `WHERE` lo más cerca posible de la fuente) y *projection pruning* (elimina columnas no necesarias).
4. **Physical Plan**: Selecciona la estrategia de ejecución más eficiente (sort, hash, broadcast).

En nuestro Q1, Catalyst empuja el filtro `categoria = 'Tecnologia' AND monto > 300000` antes del ordenamiento, reduciendo los datos que se procesan en etapas posteriores.

## Insight → Possible Decision
- **Q1**: Identifica productos de alto valor en Tecnología → decisión de stock prioritario.
- **Q2**: Revela qué categorías generan mayor ingreso → asignación de presupuesto de marketing.
- **Q3**: Muestra las sucursales top → potencial benchmark de mejores prácticas hacia sucursales de menor rendimiento.

In [ ]:
spark.stop()
print("Sesión Spark cerrada.")